In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import json
import ast
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

from forge_net.data.dataloaders import GetSingleStepDataLoaders
from forge_net.utils.common import actions_from_feature_map
from forge_net.utils.common import MeshContainer, meshcontainer_to_volume
from forge_net.utils.math import *
import meshio
pv.start_xvfb()

In [ ]:
def process_series(args):
    """Process a single series - this will run in parallel
    
    Args:
        args: Tuple of (series_id, group_df, total_points, press_width, mask_points, seed)
    """
    series_id, group_df, total_points, press_width, mask_points, seed = args
    series_coords_t = []
    series_coords_tp1 = []
    series_steps = []
    series_positions = []
    series_rotations = []
    pv_meshes = []
    pv_meshes_tp1 = []
    bary_coords_list = []
    tri_ids_list = []
    
    for i in range(len(group_df) - 1):
        row_t = group_df.iloc[i]
        row_tp1 = group_df.iloc[i + 1]
       

        # Input mesh (coords from frame i)
        num_steps_t = row_t["solver_steps"]
        vertices_t = np.array(ast.literal_eval(row_t["vertices"])).reshape(-1,3)
        triangles_t = np.array(ast.literal_eval(row_t["triangles"])).reshape(-1,4)
        vertex_temps_t = np.array(ast.literal_eval(row_t["input_temperature"])).reshape(-1,1)
        print(np.array(vertices_t).reshape(-1,3).shape, np.array(triangles_t).reshape(-1,4).shape)

        tmp_mesh_t = meshio.Mesh(
            points=vertices_t, 
            cells=[("tetra", triangles_t)]
        )
        pv_mesh_t = pv.from_meshio(tmp_mesh_t)
        # pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_t)
        # pl.screenshot("tmp.png")

        # print(tmp_mesh_t)
        # print(type(pv_mesh_t)) 

        num_steps_tp1 = row_tp1["solver_steps"]
        vertices_tp1 = np.array(ast.literal_eval(row_tp1["vertices"])).reshape(-1,3)
        triangles_tp1 = np.array(ast.literal_eval(row_tp1["triangles"])).reshape(-1,4)
        vertex_temps_tp1 = np.array(ast.literal_eval(row_tp1["input_temperature"])).reshape(-1,1)
        tmp_mesh_tp1 = meshio.Mesh(
            points=vertices_tp1, 
            cells=[("tetra", triangles_tp1)]
        )
        pv_mesh_tp1 = pv.from_meshio(tmp_mesh_tp1)
 
        # p_tp1 = ((row_tp1["x_max_band"] + row_tp1["x_min_band"]) / 2, 0 , 0)
        # r_tp1 = eulerxyz_to_quat((row_tp1["rotation_euler_x"], 0, 0)) #-> quaternion
        # pv_mesh_t.points = transform_points(np.array(pv_mesh_t.points), np.array(r_tp1), np.array(p_tp1))
        # pv_mesh_tp1.points = transform_points(np.array(pv_mesh_tp1.points), np.array(r_tp1), np.array(p_tp1))

        sampled_points_t, point_triangle_ids, bary_coords, sampled_temps_t = tetrahedral_barycentric_sampling(
                                                                                                            pv_mesh_t, 
                                                                                                            total_points, 
                                                                                                            node_features=vertex_temps_t, 
                                                                                                            seed=seed)
        print("sampled barycenters:" , sampled_points_t)
        tri_ids_list.append(point_triangle_ids)
        bary_coords_list.append(bary_coords)

        sampled_points_tp1, sampled_temps_tp1 = update_tetrahedral_barycentric_points(
                                                                                deformed_mesh=pv_mesh_tp1, 
                                                                                tet_ids=point_triangle_ids, 
                                                                                barycentric_coords=bary_coords, 
                                                                                node_features=vertex_temps_tp1)
        
        pl = pv.Plotter()
        pl.add_mesh(pv_mesh_tp1)
        pl.show_grid()
        pl.screenshot("tmp_mesh.png")

        pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_tp1,style='wireframe')
        point_cloud = pv.PolyData(sampled_points_tp1)
        point_cloud["temps"] = sampled_temps_t
        pl.add_mesh(point_cloud,
                    scalars='temps',
                    point_size=5.0,render_points_as_spheres=True)
        pl.show_grid()
        pl.screenshot("tmp.png")
        # pl.export_html("tmp.html")

        pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_tp1,style='wireframe')
        point_cloud = pv.PolyData(sampled_points_tp1)
        point_cloud["temps"] = sampled_temps_tp1
        pl.add_mesh(point_cloud,
                    scalars='temps',
                    point_size=5.0,render_points_as_spheres=True)
        pl.show_grid()
        pl.screenshot("tmp_tp1.png")
        # pl.export_html("tmp_tp1.html")


        

In [3]:
# series_id, group_df, total_points, press_width, mask_points, seed = args
db_path="/local/scratch/groves/jax-forgeRL/JAX-FORGE/Agility_Forge_data/data/forge_database.db"
lines=1_000
total_points=10_000
mask_points=False
seed=None
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM hits LIMIT {int(lines)};", conn)
conn.close()

# Prepare arguments for each series
press_width = 1.0
series_ids = df['series_id'].unique()
args_list = [
    (series_id, 
        df[df['series_id'] == series_id].reset_index(drop=True), 
        total_points, 
        press_width,
        mask_points,
        seed
    )
    for series_id in series_ids
]

In [4]:
process_series(args_list[0])

(7531, 3) (30790, 4)
sampled barycenters: [[ 7.87643499  1.01991203 -2.98077575]
 [ 9.45424672  4.03451314  2.60542235]
 [ 4.53077178  3.26657875  1.20521877]
 ...
 [ 2.78744015 -2.21480069  7.18157674]
 [92.81295441  4.78953683  5.82106195]
 [48.07870235  7.27510419 -0.50844191]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.35031653  1.70772658 -2.31494673]
 [ 9.04718387  3.28087802  3.08877662]
 [ 4.72038142  2.34071085  1.29486975]
 ...
 [83.37029811 -2.33438767 -6.45109261]
 [51.43684785  4.60458406  6.33532566]
 [-3.75395801  3.59343423  0.49753852]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.74833802  0.84714767 -3.10306846]
 [ 9.04769245  3.99392225  2.84740702]
 [ 4.70978857  3.62950635  0.2799516 ]
 ...
 [ 5.46520177  3.64520004 -5.7201293 ]
 [58.8757875   5.9948359  -2.18415298]
 [28.39716628 -3.24912735  4.80056246]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.6468116   0.89052002 -3.65839364]
 [ 8.98749654  4.00027199  2.68559988]
 [ 4.78134684  3.48919084  1.12855

ValueError: repeats may not contain negative values.